In [1]:
#import packages
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pymannkendall as mk
import cartopy.crs as ccrs
import time
import matplotlib as mpl
from scipy.stats import linregress
# import dask
import matplotlib.ticker as mticker
import string
letters=[]
for letter in string.ascii_lowercase[0:24]:
    letters.append(letter+". ")
from functions import *

In [55]:
#read era5 surface variables
eransr=xr.open_dataset('era5/era5_surface_net_solar_radiation_historical_regridcon.nc')
eransr_21=eransr.ssr.rename({'lon':'longitude','lat':'latitude', 'valid_time':'time'}).sel(time=slice('1950-01-01','2021-12-31'))
eransr_land=(landmask(rotlon_180(cutmidlat(eransr_21))))#.rename({'valid_time':'time'}))

eradsr=xr.open_dataset('era5/era5_surface_solar_radiation_downwards_historical_regridcon.nc')
eradsr_21=eradsr.ssrd.rename({'lon':'longitude','lat':'latitude', 'valid_time':'time'}).sel(time=slice('1950-01-01','2021-12-31'))
eradsr_land=(landmask(rotlon_180(cutmidlat(eradsr_21))))

erasnw=xr.open_dataset('era5/era5_snow_depth_historical_regridcon.nc')
erasnw_21=erasnw.sd.rename({'lon':'longitude','lat':'latitude', 'valid_time':'time'}).sel(time=slice('1950-01-01','2021-12-31'))
erasnw_land=(landmask(rotlon_180(cutmidlat(erasnw_21))))

eravsw=xr.open_dataset('era5/era5_volumetric_soil_water_layer_1_historical_regridcon.nc')
eravsw_21=eravsw.swvl1.rename({'lon':'longitude','lat':'latitude', 'valid_time':'time'}).sel(time=slice('1950-01-01','2021-12-31'))
eravsw_land=(landmask(rotlon_180(cutmidlat(eravsw_21))))*1000 * 0.07 #volumetric soil watwe m3/m3 density kg/m3 depth m


erasnw_land_or=xr.where(erasnw_land>0.1,0.1,erasnw_land)
erausr_land=eradsr_land - eransr_land
eraalbedo=erausr_land/eradsr_land

In [56]:
#compute era5 slopes
varvals=[eraalbedo, erasnw_land_or, eravsw_land]
slopes=[]
pvs=[]
for i,d in enumerate(varvals):
#     if i<3:
#         dat=1/86400 * d
#     else:
    dat=d
    lh_seas=dat.groupby('time.season')
    # titles=lh_seas.mean().season.values
    # year= np.arange(1950,2025)
    titles=lh_seas.mean().season.values

    lh_dy_seas=xr.full_like(dat.groupby('time.year').mean().expand_dims(season=titles).copy(),np.nan)

    for i,ss in enumerate(titles):
        lhdy=lh_seas[ss].groupby('time.year').mean()
        lh_dy_seas[i,]=lhdy

    slope_era, pval_era = xr.apply_ufunc(
        manntrend,
        lh_dy_seas,  # your variable (time on first axis)
        input_core_dims=[['year']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float]
    )
    slopes.append(slope_era)
    pvs.append(pval_era)


/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/pymannkendall/pymannkendall.py:155: RuntimeWarning: invalid value encountered in subtract
  d[idx : idx + len(j)] = (x[j] - x[i]) / (j - i)
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1217: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a, func=_nanmedian, keepdims=keepdims,


In [57]:
#save files
slopes[0].to_netcdf('mk_trends_albedo_era.nc')
slopes[1].to_netcdf('mk_trends_snowdepth_era.nc')
slopes[2].to_netcdf('mk_trends_soilmoisture_era.nc')
pvs[0].to_netcdf('mk_pvs_albedo_era.nc')
pvs[1].to_netcdf('mk_pvs_snowdepth_era.nc')
pvs[2].to_netcdf('mk_pvs_soilmoisture_era.nc')

In [4]:
#read rsus
models=['ACCESS-CM2', 'AWI-CM-1-1-MR', 'BCC-CSM2-MR','CAMS-CSM1-0', 'CanESM5-CanOE', 'CESM2', 'CMCC-ESM2', 'CMCC-CM2-SR5', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'E3SM-1-1', 'FGOALS-g3', 'IITM-ESM',  'MIROC6', 'MPI-ESM1-2-LR', 'NESM3', 'NorESM2-MM']
to_con1=[]
to_con2=[]
to_con_rsus=[]
for i,mod in enumerate(models):
    dat=xr.open_dataset('cmip6/rsus_Amon_'+mod+'_historical_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat1=landmask(tmp_midl)
    tmp_midlat1=tmp_midlat1.drop_duplicates(dim="time")
    if i>0:
        tmp_midlat1['time']=to_con1[0].time.values
    to_con1.append(tmp_midlat1)
   # temp_datcut[mod]=temp_dat#.sel(time=(slice('1950-01-01','2024-12-31')))

    dat=xr.open_dataset('cmip6/rsus_Amon_'+mod+'_ssp585_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat2=landmask(tmp_midl)
    tmp_midlat2=tmp_midlat2.drop_duplicates(dim="time")
    if i>0:
        tmp_midlat2['time']=to_con2[0].time.values
    to_con2.append(tmp_midlat2) 
    
    tmp_con=xr.concat([tmp_midlat1,tmp_midlat2], dim='time')
    to_con_rsus.append(tmp_con)

In [5]:
#read rsds
models=['ACCESS-CM2', 'AWI-CM-1-1-MR', 'BCC-CSM2-MR','CAMS-CSM1-0', 'CanESM5-CanOE', 'CESM2', 'CMCC-ESM2', 'CMCC-CM2-SR5', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'E3SM-1-1', 'FGOALS-g3', 'IITM-ESM',  'MIROC6', 'MPI-ESM1-2-LR', 'NESM3', 'NorESM2-MM']
to_con1=[]
to_con2=[]
to_con_rsds=[]
for i,mod in enumerate(models):
    dat=xr.open_dataset('cmip6/rsds_Amon_'+mod+'_historical_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat1=landmask(tmp_midl)
    tmp_midlat1=tmp_midlat1.drop_duplicates(dim="time")
    if i>0:
        tmp_midlat1['time']=to_con1[0].time.values
    to_con1.append(tmp_midlat1)
   # temp_datcut[mod]=temp_dat#.sel(time=(slice('1950-01-01','2024-12-31')))

    dat=xr.open_dataset('cmip6/rsds_Amon_'+mod+'_ssp585_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat2=landmask(tmp_midl)
    tmp_midlat2=tmp_midlat2.drop_duplicates(dim="time")
    if i>0:
        tmp_midlat2['time']=to_con2[0].time.values
    to_con2.append(tmp_midlat2) 
    
    tmp_con=xr.concat([tmp_midlat1,tmp_midlat2], dim='time')
    to_con_rsds.append(tmp_con)

In [31]:
#read snd
models=[ 'BCC-CSM2-MR', 'CanESM5', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-ESM2', 'CMCC-CM2-SR5', 'EC-Earth3-CC', 'EC-Earth3-Veg-LR', 'GFDL-ESM4', 'IPSL-CM6A-LR', 'KIOST-ESM', 'MIROC6','MRI-ESM2-0', 'NorESM2-MM', 'TaiESM1']
to_con1=[]
to_con2=[]
to_con_snd=[]
for i,mod in enumerate(models):
    dat=xr.open_dataset('cmip6/snd_LImon_'+mod+'_historical_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat1=landmask(tmp_midl)
    tmp_midlat1=tmp_midlat1.drop_duplicates(dim="time")
    if i>0:
        tmp_midlat1['time']=to_con1[0].time.values
    to_con1.append(tmp_midlat1)
   # temp_datcut[mod]=temp_dat#.sel(time=(slice('1950-01-01','2024-12-31')))

    dat=xr.open_dataset('cmip6/snd_LImon_'+mod+'_ssp585_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat2=landmask(tmp_midl)
    tmp_midlat2=tmp_midlat2.drop_duplicates(dim="time")
    if i==14 or i==15:
        new_data = tmp_midlat2.isel(time=0)
        new_data['time'].values=to_con2[0].time.values[0]
        tmp_midlat2=xr.concat([new_data,tmp_midlat2],dim='time')
    if i>0:
        tmp_midlat2['time']=to_con2[0].time.values
    to_con2.append(tmp_midlat2) 
    
    tmp_con=xr.concat([tmp_midlat1,tmp_midlat2], dim='time')
    to_con_snd.append(tmp_con)

In [47]:
#read mrsos
models=['BCC-CSM2-MR','CAMS-CSM1-0', 'CanESM5-CanOE', 'CESM2-WACCM','CESM2','CMCC-CM2-SR5','CMCC-ESM2','CNRM-CM6-1-HR','E3SM-1-1','EC-Earth3-CC','EC-Earth3-Veg-LR', 'MIROC6',  'MIROC-ES2L', 'MPI-ESM1-2-LR','MRI-ESM2-0','NorESM2-LM','NorESM2-MM','TaiESM1']
to_con1=[]
to_con2=[]
to_con_mrsos=[]
for i,mod in enumerate(models):
    dat=xr.open_dataset('cmip6/mrsos_Lmon_'+mod+'_historical_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat1=landmask(tmp_midl)
    tmp_midlat1=tmp_midlat1.drop_duplicates(dim="time")
    if i>0:
        tmp_midlat1['time']=to_con1[0].time.values
    to_con1.append(tmp_midlat1)
   # temp_datcut[mod]=temp_dat#.sel(time=(slice('1950-01-01','2024-12-31')))

    dat=xr.open_dataset('cmip6/mrsos_Lmon_'+mod+'_ssp585_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat2=landmask(tmp_midl)
    tmp_midlat2=tmp_midlat2.drop_duplicates(dim="time")
    if i>14:
        new_data = tmp_midlat2.isel(time=0)
        new_data['time'].values=to_con2[0].time.values[0]
        tmp_midlat2=xr.concat([new_data,tmp_midlat2],dim='time')
    if i>0:
        tmp_midlat2['time']=to_con2[0].time.values
    to_con2.append(tmp_midlat2) 
    
    tmp_con=xr.concat([tmp_midlat1,tmp_midlat2], dim='time')
    try:
        to_con_mrsos.append(tmp_con.drop_vars('depth'))
    except:
        to_con_mrsos.append(tmp_con)

In [36]:
#read snd hist
models=[ 'BCC-CSM2-MR', 'CanESM5', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-ESM2', 'CMCC-CM2-SR5', 'EC-Earth3-CC', 'EC-Earth3-Veg-LR', 'GFDL-ESM4', 'IPSL-CM6A-LR', 'KIOST-ESM', 'MIROC6','MRI-ESM2-0', 'NorESM2-MM', 'TaiESM1']
to_con1=[]
to_con2=[]
to_con_snd=[]
for i,mod in enumerate(models):
    dat=xr.open_dataset('cmip6/snd_LImon_'+mod+'_historical_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat1=landmask(tmp_midl)
    tmp_midlat1=tmp_midlat1.drop_duplicates(dim="time")
    if i>0:
        tmp_midlat1['time']=to_con_snd[0].time.values
   # temp_datcut[mod]=temp_dat#.sel(time=(slice('1950-01-01','2024-12-31')))

   
    to_con_snd.append(tmp_midlat1)

In [37]:
#read mrsos hist
models=['BCC-CSM2-MR','CAMS-CSM1-0', 'CanESM5-CanOE', 'CESM2-WACCM','CESM2','CMCC-CM2-SR5','CMCC-ESM2','CNRM-CM6-1-HR','E3SM-1-1','EC-Earth3-CC','EC-Earth3-Veg-LR', 'MIROC6',  'MIROC-ES2L', 'MPI-ESM1-2-LR','MRI-ESM2-0','NorESM2-LM','NorESM2-MM','TaiESM1']
to_con1=[]
to_con2=[]
to_con_mrsos=[]
for i,mod in enumerate(models):
    dat=xr.open_dataset('cmip6/mrsos_Lmon_'+mod+'_historical_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat1=landmask(tmp_midl)
    tmp_midlat1=tmp_midlat1.drop_duplicates(dim="time")
    if i>0:
        tmp_midlat1['time']=to_con_mrsos[0].time.values

   # temp_datcut[mod]=temp_dat#.sel(time=(slice('1950-01-01','2024-12-31')))

   
    try:
        to_con_mrsos.append(tmp_midlat1.drop_vars('depth'))
    except:
        to_con_mrsos.append(tmp_midlat1)

In [38]:
#concat models
# rsus_cmip=xr.concat(to_con_rsus, dim='model')
# rsds_cmip=xr.concat(to_con_rsds, dim='model')
# rnet=rsds_cmip.rsds- rsus_cmip.rsus
cmip_albedo=rsus_cmip.rsus/rsds_cmip.rsds

snw_cmip=xr.concat(to_con_snd, dim='model')
vsw_cmip=xr.concat(to_con_mrsos, dim='model')

In [39]:
#compute CMIP6 surface var trends
snw_cmip_or=xr.where(snw_cmip.snd>0.1,0.1,snw_cmip.snd)
cmipvars=[cmip_albedo, snw_cmip_or,vsw_cmip.mrsos]
slopes_cmp=[]
pvs_cmp=[]
for d in cmipvars:
    dat=d
    lh_seas=dat.groupby('time.season')
    # titles=lh_seas.mean().season.values
    # year= np.arange(1950,2025)
    titles=lh_seas.mean().season.values

    lh_dy_seas=xr.full_like(dat.groupby('time.year').mean().expand_dims(season=titles).copy(),np.nan)

    for i,ss in enumerate(titles):
        lhdy=lh_seas[ss].groupby('time.year').mean()
        lh_dy_seas[i,]=lhdy

    slope_cmip, pval_cmip = xr.apply_ufunc(
        manntrend,
        lh_dy_seas,  # your variable (time on first axis)
        input_core_dims=[['year']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float]
    )
    slopes_cmp.append(slope_cmip)
    pvs_cmp.append(pval_cmip)

# slopes.append(slope)
# pvs.append(pval)
# lh_dy_seas=lh_dy_seas.to_dataset(name='lh')

/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/pymannkendall/pymannkendall.py:155: RuntimeWarning: invalid value encountered in subtract
  d[idx : idx + len(j)] = (x[j] - x[i]) / (j - i)
/home/rwhite/ladmasu/mambaforge/envs/comput/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1217: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a, func=_nanmedian, keepdims=keepdims,


In [40]:
#save files
slopes_cmp[0].to_netcdf('mk_trends2014_albedo_cmip.nc')
slopes_cmp[1].to_netcdf('mk_trends2014_snowdepth_cmip.nc')
slopes_cmp[2].to_netcdf('mk_trends2014_soilmoisture_cmip.nc')
pvs_cmp[0].to_netcdf('mk_pvs_albedo2014_cmip.nc')
pvs_cmp[1].to_netcdf('mk_pvs_snowdepth2014_cmip.nc')
pvs_cmp[2].to_netcdf('mk_pvs_soilmoisture2014_cmip.nc')

In [17]:
#read hfls
models1=['ACCESS-CM2','AWI-CM-1-1-MR','BCC-CSM2-MR','CESM2','CMCC-ESM2','CNRM-ESM2-1','EC-Earth3-CC','GFDL-ESM4','IITM-ESM','INM-CM5-0','MIROC6','MPI-ESM1-2-LR','MRI-ESM2-0','NESM3','NorESM2-MM','TaiESM1']# temp_datcut={}
models22=['CESM2-WACCM','CNRM-CM6-1','INM-CM4-8','IPSL-CM6A-LR', 'MIROC-ES2L','CNRM-CM6-1-HR']
models_hfls=np.array(models1+models22)
to_con1=[]
to_con2=[]
to_con_hfls=[]
for i,mod in enumerate(models_hfls):
    dat=xr.open_dataset('cmip6/hfls_Amon_'+mod+'_historical_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})

    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat1=landmask(tmp_midl)
    tmp_midlat1=tmp_midlat1.drop_duplicates(dim="time")
    if i>0:
        tmp_midlat1['time']=to_con_hfss[0].time.values
    to_con_hfls.append(tmp_midlat1)
#     try:
#         to_con_hfss.append(tmp_midlat1.drop_vars('depth'))
#     except:
#         to_con_hfss.append(tmp_midlat1)

In [18]:
#read hfss
models1=['ACCESS-CM2','AWI-CM-1-1-MR', 'CanESM5','CESM2','CMCC-ESM2','CNRM-ESM2-1','GFDL-ESM4','IITM-ESM','INM-CM5-0','KIOST-ESM','MIROC6','MPI-ESM1-2-LR','MRI-ESM2-0','NESM3','NorESM2-MM','TaiESM1']# temp_datcut={}
models22=['CESM2-WACCM', 'CMCC-CM2-SR5', 'CNRM-CM6-1','INM-CM4-8','IPSL-CM6A-LR', 'MIROC-ES2L','CNRM-CM6-1-HR']
models_hfss=np.array(models1+models22)
to_con1=[]
to_con2=[]
to_con_hfss=[]
for i,mod in enumerate(models_hfss):
    dat=xr.open_dataset('cmip6/hfss_Amon_'+mod+'_historical_regridcon.nc')#.rename({'lat':'latitude','lon':'longitude'})
    dat1=dat.convert_calendar('noleap').rename({'lon':'longitude','lat':'latitude'})
    temp_dat=rotlon_180(dat1)
    tmp1=temp_dat.sortby(temp_dat.time)
    tmp_midl=cutmidlat(tmp1)
    tmp_midlat1=landmask(tmp_midl)
    tmp_midlat1=tmp_midlat1.drop_duplicates(dim="time")
    if i>0:
        tmp_midlat1['time']=to_con_hfss[0].time.values
    to_con_hfss.append(tmp_midlat1)

In [19]:
#concatenate models
rsus_cmip=xr.concat(to_con_rsus, dim='model')
rsds_cmip=xr.concat(to_con_rsds, dim='model')
hfls_cmip=xr.concat(to_con_hfls, dim='model')
hfss_cmip=xr.concat(to_con_hfss, dim='model')

In [21]:
#compute CMIP6 flux trends

cmipvars=[rsus_cmip.rsus, rsds_cmip.rsds,hfls_cmip.hfls,hfss_cmip.hfss]
slopes_cmp=[]
pvs_cmp=[]
for d in cmipvars:
    dat=d
    lh_seas=dat.groupby('time.season')
    # titles=lh_seas.mean().season.values
    # year= np.arange(1950,2025)
    titles=lh_seas.mean().season.values

    lh_dy_seas=xr.full_like(dat.groupby('time.year').mean().expand_dims(season=titles).copy(),np.nan)

    for i,ss in enumerate(titles):
        lhdy=lh_seas[ss].groupby('time.year').mean()
        lh_dy_seas[i,]=lhdy

    slope_cmip, pval_cmip = xr.apply_ufunc(
        manntrend,
        lh_dy_seas,  # your variable (time on first axis)
        input_core_dims=[['year']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float]
    )
    slopes_cmp.append(slope_cmip)
    pvs_cmp.append(pval_cmip)

# slopes.append(slope)
# pvs.append(pval)
# lh_dy_seas=lh_dy_seas.to_dataset(name='lh')

In [22]:
#save files
slopes_cmp[0].to_netcdf('mk_trends_rsus2014_cmip.nc')
slopes_cmp[1].to_netcdf('mk_trends_rsds2014_cmip.nc')
slopes_cmp[2].to_netcdf('mk_trends_hfls2014_cmip.nc')
slopes_cmp[3].to_netcdf('mk_trends_hfss2014_cmip.nc')
pvs_cmp[0].to_netcdf('mk_pvs_rsus2014_cmip.nc')
pvs_cmp[1].to_netcdf('mk_pvs_rsds2014_cmip.nc')
pvs_cmp[2].to_netcdf('mk_pvs_hfls2014_cmip.nc')
pvs_cmp[3].to_netcdf('mk_pvs_hfss2014_cmip.nc')